# Simple Health Care Chatbot for Google Colab

This notebook builds a beginner-friendly health care chatbot in plain Python.

**What it can do**
- Answer common health questions with short, safety-focused guidance.
- Recognize urgent warning signs and tell the user to seek emergency care.
- Suggest basic self-care steps for minor symptoms.
- Remind users that it is not a doctor and cannot diagnose disease.

> **Important:** This chatbot is for education only. It does not replace a licensed doctor, nurse, pharmacist, or emergency service. If symptoms are severe, sudden, worsening, or life-threatening, call your local emergency number immediately.


In [ ]:
import re
import textwrap

print('✅ Health care chatbot is ready to load.')


## Chatbot Code
Run the cell below. The code is intentionally simple: it uses keyword matching, a small medical knowledge base, and safety rules.


In [ ]:
DISCLAIMER = (
    "I can share general health education, but I cannot diagnose or replace a clinician. "
    "For personal medical advice, contact a doctor, nurse, or pharmacist."
)

EMERGENCY_KEYWORDS = {
    "chest pain": "Chest pain can be serious, especially with shortness of breath, sweating, nausea, or pain spreading to the arm/jaw/back.",
    "can't breathe": "Trouble breathing can become life-threatening.",
    "cannot breathe": "Trouble breathing can become life-threatening.",
    "shortness of breath": "Shortness of breath can be serious if severe, sudden, or new.",
    "stroke": "Stroke symptoms need emergency care quickly.",
    "face drooping": "Face drooping can be a stroke warning sign.",
    "weakness on one side": "One-sided weakness can be a stroke warning sign.",
    "suicidal": "Feeling suicidal is an emergency and you deserve immediate support.",
    "overdose": "Possible overdose needs urgent medical help.",
    "severe bleeding": "Severe bleeding needs urgent medical attention.",
    "unconscious": "Unconsciousness is an emergency.",
    "seizure": "A first seizure, long seizure, or repeated seizures need urgent care.",
    "severe allergic reaction": "Severe allergic reactions can block breathing.",
    "anaphylaxis": "Anaphylaxis is a medical emergency.",
}

HEALTH_TOPICS = {
    "fever": {
        "answer": "A fever is a body temperature higher than normal, often caused by infection. Rest, drink fluids, and consider acetaminophen/paracetamol or ibuprofen if you can take them safely.",
        "care": "Seek medical care for fever in a baby under 3 months, fever lasting more than 3 days, very high fever, stiff neck, confusion, rash, dehydration, breathing trouble, or if you are immunocompromised."
    },
    "cough": {
        "answer": "A cough is commonly caused by a cold, flu, allergies, asthma, reflux, or other respiratory irritation. Warm fluids, honey for adults and children over 1 year, and avoiding smoke may help.",
        "care": "Get medical advice if cough lasts more than 3 weeks, produces blood, causes chest pain, comes with high fever, wheezing, shortness of breath, or affects a young child or frail adult."
    },
    "sore throat": {
        "answer": "A sore throat is often viral and improves with fluids, rest, salt-water gargles, and pain relief medicine if safe for you.",
        "care": "See a clinician for trouble breathing or swallowing, drooling, severe one-sided throat pain, rash, fever, swollen neck glands, or symptoms lasting more than a few days."
    },
    "headache": {
        "answer": "Common headaches can be related to stress, dehydration, poor sleep, eyestrain, sinus problems, or migraine. Rest, water, a quiet dark room, and safe pain relief may help.",
        "care": "Seek urgent care for sudden worst headache, headache after head injury, fever with stiff neck, confusion, weakness, vision loss, pregnancy, or a new severe headache after age 50."
    },
    "stomach pain": {
        "answer": "Mild stomach pain may come from indigestion, gas, constipation, viral illness, or food irritation. Sip fluids and eat bland foods if tolerated.",
        "care": "Get urgent help for severe or worsening pain, pain with a hard belly, fainting, blood in stool/vomit, persistent vomiting, pregnancy, or right-lower belly pain."
    },
    "diarrhea": {
        "answer": "Diarrhea is often caused by infection or food irritation. The main treatment is preventing dehydration with water or oral rehydration solution.",
        "care": "Call a clinician for blood in stool, high fever, severe dehydration, severe belly pain, diarrhea lasting more than 2-3 days, or diarrhea in babies, older adults, or high-risk patients."
    },
    "vomiting": {
        "answer": "Vomiting can happen with stomach viruses, food poisoning, migraine, pregnancy, medications, or other conditions. Take small sips of fluid frequently and avoid heavy foods at first.",
        "care": "Seek care for signs of dehydration, blood or green vomit, severe headache, stiff neck, severe belly pain, chest pain, confusion, or vomiting that does not stop."
    },
    "cold": {
        "answer": "The common cold usually improves in 7-10 days. Rest, fluids, saline spray, honey for cough if age over 1 year, and safe pain relief can help symptoms.",
        "care": "Get medical advice for breathing trouble, chest pain, symptoms lasting over 10 days without improvement, dehydration, high fever, or high-risk health conditions."
    },
    "flu": {
        "answer": "Flu can cause fever, cough, sore throat, body aches, chills, and tiredness. Rest and fluids are important; antiviral medicine works best when started early for high-risk people.",
        "care": "Seek care urgently for breathing trouble, chest pain, confusion, dehydration, blue lips, severe weakness, or symptoms that improve then suddenly worsen."
    },
    "covid": {
        "answer": "COVID-19 symptoms can include fever, cough, sore throat, congestion, loss of taste or smell, tiredness, and body aches. Testing, staying home when contagious, masking, fluids, and rest may help reduce spread and support recovery.",
        "care": "Seek urgent care for trouble breathing, chest pain, confusion, bluish lips/face, severe weakness, dehydration, or if you are high risk and may need antiviral treatment."
    },
    "allergy": {
        "answer": "Allergies can cause sneezing, itchy eyes, runny nose, rash, or hives. Avoid triggers when possible; antihistamines or nasal steroid sprays may help if safe for you.",
        "care": "Call emergency services for swelling of the lips/tongue/throat, trouble breathing, wheezing, fainting, or widespread hives after a trigger."
    },
    "burn": {
        "answer": "For a minor burn, cool the area under running cool water for about 20 minutes, remove tight items, cover with a clean non-stick dressing, and do not apply ice or butter.",
        "care": "Get urgent care for large, deep, chemical, electrical, face/hand/genital burns, burns in babies/older adults, or signs of infection."
    },
    "cut": {
        "answer": "For a small cut, wash hands, apply gentle pressure to stop bleeding, rinse with clean water, apply a clean dressing, and keep it clean.",
        "care": "Seek care for deep wounds, animal/human bites, dirty wounds, numbness, gaping edges, uncontrolled bleeding, or if tetanus vaccination is not up to date."
    },
    "blood pressure": {
        "answer": "Blood pressure measures force in arteries. A healthy lifestyle, less salt, regular activity, healthy weight, no smoking, and prescribed medicine can help control high blood pressure.",
        "care": "Very high readings with chest pain, shortness of breath, severe headache, confusion, weakness, or vision changes need urgent medical care."
    },
    "diabetes": {
        "answer": "Diabetes means blood sugar is higher than normal. Management often includes healthy eating, physical activity, glucose monitoring, and medication or insulin when prescribed.",
        "care": "Seek urgent care for confusion, fainting, severe weakness, very high or very low sugar, vomiting, dehydration, fruity breath, or rapid breathing."
    },
    "medicine": {
        "answer": "Take medicines exactly as prescribed or as the label says. Check dose, timing, allergies, interactions, pregnancy safety, and whether to take with food.",
        "care": "Ask a pharmacist or clinician before mixing medicines, giving medicine to a child, using medicine while pregnant, or if you have kidney/liver disease. For overdose, call emergency services or poison control."
    },
}

GREETINGS = {"hi", "hello", "hey", "good morning", "good afternoon", "good evening"}
GOODBYES = {"bye", "exit", "quit", "goodbye"}


def clean_text(text):
    return re.sub(r"\s+", " ", text.lower().strip())


def find_topic(user_text):
    for topic in HEALTH_TOPICS:
        if topic in user_text:
            return topic
    return None


def check_emergency(user_text):
    for phrase, reason in EMERGENCY_KEYWORDS.items():
        if phrase in user_text:
            return phrase, reason
    return None, None


def health_chatbot(message):
    text = clean_text(message)

    if not text:
        return "Please type a health question, symptom, or topic. " + DISCLAIMER

    if text in GOODBYES:
        return "Take care. If you feel seriously unwell, seek medical help promptly."

    if text in GREETINGS:
        return "Hello! Tell me your symptom or health question. " + DISCLAIMER

    phrase, reason = check_emergency(text)
    if phrase:
        return (
            f"🚨 Possible emergency: {reason} "
            "Please call your local emergency number now or go to the nearest emergency department. "
            "If you are in the U.S. and this is a mental health crisis, call or text 988."
        )

    topic = find_topic(text)
    if topic:
        info = HEALTH_TOPICS[topic]
        return f"{info['answer']} {info['care']} {DISCLAIMER}"

    return (
        "I do not have a specific answer for that topic. Please describe the main symptom, how long it has been happening, "
        "your age group, and any warning signs such as severe pain, trouble breathing, confusion, fainting, bleeding, or dehydration. "
        + DISCLAIMER
    )

print('✅ Chatbot functions loaded. Try asking about fever, cough, headache, diabetes, blood pressure, burns, cuts, allergies, flu, COVID, or medicine safety.')


## Try One Question
Run this cell and type one health question.


In [ ]:
question = input('Ask a health question: ')
print('\nBot:')
print(textwrap.fill(health_chatbot(question), width=100))


## Continuous Chat Mode
Run this cell to keep chatting. Type `bye`, `exit`, or `quit` to stop.


In [ ]:
print('🤖 Health Care Chatbot is running. Type bye, exit, or quit to stop.')

while True:
    user_message = input('You: ')
    bot_message = health_chatbot(user_message)
    print('Bot:', textwrap.fill(bot_message, width=100))
    if user_message.strip().lower() in GOODBYES:
        break


## Example Questions
Try these examples:

- `I have a fever`
- `What should I do for a cough?`
- `I have chest pain and shortness of breath`
- `How can I treat a small burn?`
- `Tell me about diabetes`
- `Is it safe to mix medicines?`
